## Mall Customers Clustering with KMeans, DBSCAN, & Agglomerative - Unsupervised Learning

> Tugas ini bertujuan untuk membangun model clustering pelanggan mall dengan KMeans, DBSCAN, dan Agglomerative. Dataset berasal dari sini : https://gist.githubusercontent.com/pravalliyaram/5c05f43d2351249927b8a3f3cc3e5ecf/raw/Mall_Customers.csv

> Referensi : https://colab.research.google.com/drive/1H1CGYKyAQ1mjy6vDAzhgeVctEOOeLPqC?usp=sharing


### 1. Import module yang diperlukan

In [ ]:
# 1. Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score

### TODO #1 :

> Isi insight : Sebenarnya, bagaimana ciri-ciri dari sebuah dataset yang bisa dipakai untuk clustering?

> Dataset yang cocok untuk clustering biasanya punya ciri-ciri sebagai berikut:
>
> 1. **Tidak punya label/target** : ini ciri utama yang membedakan dengan supervised learning. Clustering itu unsupervised, jadi tidak ada kolom "jawaban benar" yang dipakai untuk training. Tujuannya justru menemukan grup-grup tersembunyi di dalam data.
>
> 2. **Fitur numerik atau bisa dikonversi ke numerik** : algoritma clustering bekerja dengan menghitung jarak antar titik (pakai Euclidean distance, Manhattan, dll), jadi datanya harus dalam bentuk angka. Kalau ada fitur kategorikal, perlu di-encode dulu.
>
> 3. **Skala antar fitur seimbang** : kalau ada fitur dengan range besar (misal pendapatan jutaan rupiah) dan fitur lain dengan range kecil (misal usia), maka fitur yang besar akan mendominasi perhitungan jarak. Solusinya pakai scaling seperti StandardScaler atau MinMaxScaler.
>
> 4. **Ada potensi grup tersembunyi** : data yang isinya seragam (homogen) tidak akan menghasilkan cluster yang bermakna. Idealnya data punya variasi yang cukup sehingga bisa terbentuk kelompok-kelompok dengan karakteristik berbeda.
>
> 5. **Tidak terlalu banyak fitur noise** : kalau dataset punya banyak fitur yang tidak relevan, clustering bisa terganggu. Biasanya dilakukan dimensionality reduction (seperti PCA) atau seleksi fitur sebelum clustering.
>
> Contoh kasus yang cocok untuk clustering: segmentasi pelanggan, pengelompokan dokumen berdasarkan topik, pengelompokan pasien berdasarkan gejala, atau pengelompokan produk berdasarkan karakteristiknya.


### 2. Data collection

In [ ]:
# 2. Load Dataset dari GitHub Gist
url = "https://gist.githubusercontent.com/pravalliyaram/5c05f43d2351249927b8a3f3cc3e5ecf/raw/Mall_Customers.csv"
df = pd.read_csv(url)
df.head()

### 3.  Data preprocessing

In [ ]:
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### TODO #2 :

> Isi insight : Kenapa menggunakan fitur ini? Bagaimana yang lainnya? (CustomerID, Gender)

> Fitur **Age**, **Annual Income (k$)**, dan **Spending Score (1-100)** dipilih karena tiga alasan utama:
>
> 1. **Ketiganya numerik dan bermakna** : usia, pendapatan, dan score belanja punya nilai numerik yang bisa langsung dihitung jaraknya. Selain itu, ketiganya benar-benar mencerminkan profil pelanggan yang berguna untuk segmentasi.
>
> 2. **Relevan dengan tujuan bisnis** : kalau mall mau membuat strategi marketing, mereka butuh tahu siapa pelanggannya, berapa daya beli mereka, dan seberapa aktif mereka belanja. Ketiga fitur ini langsung menjawab pertanyaan tersebut.
>
> 3. **Punya variasi yang cukup** : dari data terlihat ketiga fitur ini punya range nilai yang lebar dan bervariasi, sehingga ada potensi terbentuk cluster yang jelas.
>
> Untuk fitur lainnya:
>
> - **CustomerID** : ini cuma identifier unik untuk tiap pelanggan, semacam nomor urut. Tidak ada makna analitis di dalamnya, kalau dipakai untuk clustering hasilnya tidak masuk akal karena nilai 1, 2, 3, dst tidak menunjukkan kemiripan apapun antar pelanggan. Jadi wajib di-drop.
>
> - **Gender** : sebenarnya bisa dipakai dengan encoding (Male=0, Female=1 atau pakai one-hot), tapi keputusannya tergantung tujuan analisis. Untuk segmentasi awal seperti ini, gender bukan penentu utama perilaku belanja, fitur seperti pendapatan dan spending score lebih kuat. Kalau dipaksakan dimasukkan, dimensi datanya bertambah tapi insight-nya tidak banyak berubah. Tapi kalau tujuannya analisis demografis, gender bisa relevan.
>
> Jadi pemilihan fitur untuk clustering bukan sekedar "semakin banyak fitur semakin bagus", tapi harus dipilih yang relevan dengan tujuan analisis.


### 4. KMeans Clustering (k=5)

In [ ]:
# KMeans Clustering (k=5)
kmeans = KMeans(n_clusters=5, random_state=42)
y_kmeans = kmeans.fit_predict(X_scaled)
df['Cluster_KMeans'] = y_kmeans

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster_KMeans', palette='Set1')
plt.title("KMeans Clustering (5 cluster)")
plt.show()

print("\n\nSilhouette Score KMeans:", silhouette_score(X_scaled, y_kmeans))


### TODO #3 :

> Isi insight : Bagaimana cara kerja dari K-Means Clustering? Dan apa analisismu terhadap hasil diatas?

> **Cara kerja KMeans:**
>
> 1. **Tentukan jumlah cluster (k)** : dalam kasus ini k=5, artinya kita ingin data dikelompokkan jadi 5 cluster.
>
> 2. **Inisialisasi centroid** : pilih 5 titik awal secara random sebagai pusat cluster.
>
> 3. **Assign data ke centroid terdekat** : tiap titik data dimasukkan ke cluster yang centroid-nya paling dekat (berdasarkan Euclidean distance).
>
> 4. **Update centroid** : centroid baru dihitung sebagai rata-rata posisi semua titik di cluster tersebut.
>
> 5. **Iterasi** : ulangi langkah 3 dan 4 sampai centroid tidak berubah lagi (konvergen) atau mencapai max iterasi.
>
> **Analisis hasil:**
>
> Dari scatter plot KMeans dengan k=5, terlihat bahwa data terbagi jadi 5 kelompok yang cukup jelas berdasarkan Annual Income dan Spending Score. Cluster yang terbentuk umumnya menggambarkan beberapa segmen pelanggan:
>
> - Pelanggan dengan **income tinggi tapi spending rendah** (kemungkinan pelit atau hati-hati membelanjakan uang)
> - Pelanggan dengan **income tinggi dan spending tinggi** (target premium, pelanggan ideal)
> - Pelanggan dengan **income rendah tapi spending tinggi** (impulsive buyer atau usia muda)
> - Pelanggan dengan **income rendah dan spending rendah** (segmen ekonomi rendah)
> - Pelanggan dengan **income dan spending menengah** (segmen rata-rata)
>
> Silhouette Score yang dihasilkan biasanya di sekitar 0.4, ini menunjukkan kualitas cluster yang cukup baik meskipun tidak sempurna. Hasil ini cocok untuk strategi segmentasi marketing.


### TODO #4 :

> Isi insight : Apa itu Silhouette Score?

> **Silhouette Score** adalah metrik untuk mengukur seberapa baik kualitas clustering yang dihasilkan, tanpa perlu label asli (karena clustering itu unsupervised).
>
> **Cara hitungnya untuk tiap titik data:**
>
> - Hitung **a** = rata-rata jarak titik ini ke titik-titik lain di cluster yang sama (intra-cluster distance)
> - Hitung **b** = rata-rata jarak titik ini ke titik-titik di cluster terdekat selain cluster sendiri (inter-cluster distance)
> - Silhouette score = (b - a) / max(a, b)
>
> **Range nilainya dari -1 sampai 1:**
>
> - **Mendekati 1** : titik ini sangat cocok di cluster-nya, dan jauh dari cluster lain. Ini berarti cluster terbentuk dengan baik dan terpisah jelas.
> - **Sekitar 0** : titik ini ada di "perbatasan" antara dua cluster, kurang jelas masuk yang mana.
> - **Negatif (mendekati -1)** : titik ini sebenarnya lebih cocok di cluster lain, kemungkinan masuk cluster yang salah.
>
> **Interpretasi umum:**
>
> - Score > 0.7 : struktur cluster sangat kuat
> - Score 0.5 - 0.7 : struktur cluster baik
> - Score 0.25 - 0.5 : struktur cluster cukup, tapi bisa diperbaiki
> - Score < 0.25 : struktur cluster lemah, mungkin salah jumlah cluster atau pemilihan algoritma
>
> Untuk membandingkan beberapa algoritma clustering pada data yang sama, kita bisa pakai Silhouette Score sebagai patokan. Algoritma dengan score lebih tinggi biasanya menghasilkan cluster yang lebih jelas dan terpisah.


### 5. DBSCAN Clustering

In [ ]:
# Write your code here

### TODO #5 :

> Isi insight : Bagaimana cara kerja dari DBSCAN Clustering? Dan apa analisismu terhadap hasil diatas?

> Insight

### 6. Agglomerative Clustering

In [ ]:
# Write your code here

### TODO #6 :

> Isi insight : Bagaimana cara kerja dari Agglomerative Clustering? Dan apa analisismu terhadap hasil diatas?

> Insight

### 7. Perbandingan

In [ ]:
# Write your code here

## TODO #7 **Analisis Visual: Perbandingan Hasil Clustering (3 Algoritma)**

(Isi TODO dibawah)

### **1. KMeans**

TODO

---

### **2. DBSCAN**

TODO
---

### **3. Agglomerative Clustering**

TODO

---

## Kesimpulan Naratif

TODO
